In [1]:
import os
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").appName("ews-features").getOrCreate()

rd = lambda f: spark.read.csv(f"../data/raw/{f}.csv", header=True, inferSchema=True)
info, reg, svle, vle, sa, asm = (rd(n) for n in
    ["studentInfo", "studentRegistration", "studentVle", "vle", "studentAssessment", "assessments"])

keys = ["code_module", "code_presentation", "id_student"]
info = info.withColumn("at_risk",
    F.when(F.col("final_result").isin("Fail", "Withdrawn"), 1).otherwise(0))
base = info.join(reg.select(*keys, "date_registration", "date_unregistration"), keys, "left")

cat_cols = ["code_module", "gender", "region", "highest_education",
            "imd_band", "age_band", "disability"]

In [2]:
def build_features(cutoff):
    cohort = (base.filter(F.col("date_unregistration").isNull() |
                          (F.col("date_unregistration") > cutoff))
                  .drop("date_unregistration", "final_result"))

    clk = (svle.filter(F.col("date") <= cutoff)
           .join(vle.select("id_site", "activity_type"), "id_site"))
    totals = clk.groupBy(keys).agg(
        F.sum("sum_click").alias("total_clicks"),
        F.countDistinct("date").alias("active_days"),
        F.sum(F.when(F.col("date") > cutoff - 14, F.col("sum_click")).otherwise(0)).alias("recent_clicks"))
    by_type = clk.groupBy(keys).pivot("activity_type").agg(F.sum("sum_click"))
    by_type = by_type.select(*keys, *[F.col(c).alias(f"clk_{c}") for c in by_type.columns if c not in keys])

    a = sa.join(asm, "id_assessment").filter(F.col("date_submitted") <= cutoff)
    asm_feat = a.groupBy(keys).agg(
        F.count("*").alias("n_submitted"),
        F.avg("score").alias("avg_score"),
        F.sum(F.when(F.col("date_submitted") > F.col("date"), 1).otherwise(0)).alias("n_late"))
    due = (asm.filter(F.col("date") <= cutoff).groupBy("code_module", "code_presentation")
              .agg(F.count("*").alias("n_due")))

    feat = (cohort.join(totals, keys, "left").join(by_type, keys, "left")
                  .join(asm_feat, keys, "left")
                  .join(due, ["code_module", "code_presentation"], "left"))
    clk_cols = [c for c in feat.columns if c.startswith("clk_")]
    feat = feat.fillna(0, subset=["total_clicks", "active_days", "recent_clicks",
                                  "n_submitted", "n_late", "n_due"] + clk_cols)
    feat = (feat.withColumn("n_missed", F.greatest(F.col("n_due") - F.col("n_submitted"), F.lit(0)))
                .withColumn("no_submission", F.when(F.col("n_submitted") == 0, 1).otherwise(0))
                .fillna("Unknown", subset=cat_cols))
    med = feat.approxQuantile("date_registration", [0.5], 0.01)[0]
    feat = feat.fillna({"avg_score": 0, "date_registration": med})

    num_cols = ["total_clicks", "active_days", "recent_clicks", "n_submitted", "avg_score",
                "n_missed", "n_late", "n_due", "date_registration", "studied_credits",
                "num_of_prev_attempts", "no_submission"] + clk_cols
    for c in num_cols:
        feat = feat.withColumn(c, F.col(c).cast("double"))
    return feat.withColumn("split", F.when(F.pmod(F.hash("id_student"), F.lit(10)) < 8, "train")
                                     .otherwise("test"))

In [3]:
os.makedirs("../data/processed", exist_ok=True)
for cutoff in [28, 56, 84]:
    f = build_features(cutoff)
    pdf = f.toPandas()
    pdf.to_parquet(f"../data/processed/features_day{cutoff}.parquet", index=False)
    print(cutoff, len(pdf), f"at-risk {pdf['at_risk'].mean():.1%}")

28 27538 at-risk 44.1%
56 26522 at-risk 42.0%
84 25724 at-risk 40.2%
